In [1]:
import sunpy.map
import os
import re
from datetime import datetime
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.io.fits.scripts import fitsheader
from matplotlib.colors import LogNorm, Normalize
import numpy as np
import image as img
from astropy.coordinates import SkyCoord
import astropy.units as u
from PIL import Image
from IPython.display import HTML
import time
from matplotlib.patches import Wedge, Circle
import Sunlimb
import matplotlib.animation as animation
%matplotlib notebook

In [2]:
file_path = "E:/python/projects/alfven/data/hrieuvopn/offset_data/10moving/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)

In [4]:
def draw_fig(sun_map, colorbar=False, limb=False, grid=False, rotate=False, process=None, **kwargs):
    """
    绘制sunpy.map.Map对象图像
    :param sun_map: 文件名或sunpy.map.Map对象
    :param colorbar: 是否在图像中绘制colorbar
    :param limb: 是否绘制太阳边界
    :param grid: 是否绘制网格
    :param rotate: bool，True则按照CROTA对图像进行旋转
    :param kwargs: 传递给sunpy.map.Map.plot()方法的其他参数
    :param process: 图像数据处理的函数
    :return: fig对象
    """
    if type(sun_map) == str:
        sun_map = sunpy.map.Map(sun_map)
    if rotate:
        sun_map = sun_map.rotate()
    if process is not None:
        map_data = process(sun_map.data)
        sun_map = sunpy.map.Map(map_data, sun_map.fits_header)
    fig = plt.figure()
    ax = fig.add_subplot(projection=sun_map)
    im = sun_map.plot(axes=ax, **kwargs)
    if colorbar:
        plt.colorbar(im, ax=ax)
    if limb:
        sun_map.draw_limb(axes=ax)
    if grid:
        sun_map.draw_grid(axes=ax)
    return fig

def gradiant(image):
    g_x, g_y = np.gradient(image)
    # g_data = np.sqrt(g_x**2 + g_y**2)
    g_image = g_y
    return g_image
def unsharp_mask(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    unsharp_mask_image = img.unsharp_mask(image_nan_to_0, kernel_size=7, amount=1)
    unsharp_mask_image[np.isnan(image)] = np.nan
    return image - unsharp_mask_image
def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1]+coe.data[2]
    hq = img.box_car_smoothing(hq, kernel_size=7)
    hq[np.isnan(image)] = np.nan
    return hq
def cut_map(hri, bottom_left, top_right):
    if type(hri) is str:
        hri = sunpy.map.Map(hri)
    bottom_left = SkyCoord(bottom_left[0]*u.arcsec, bottom_left[1]*u.arcsec, frame=hri.coordinate_frame)
    top_right = SkyCoord(top_right[0]*u.arcsec, top_right[1]*u.arcsec, frame=hri.coordinate_frame)
    
    submap = hri.submap(bottom_left, top_right=top_right)
    return submap

In [ ]:
data1, header = fits.getdata(files[30], header=True)
data2 = fits.getdata(files[31])

diff_data = data1 - data2
diff_map = sunpy.map.Map(diff_data, header) 
draw_fig(diff_map, colorbar=True, limb=True, norm=Normalize(vmin=-20, vmax=20), cmap='sdoaia171', process=atrous)

# 绘制梯度图

In [ ]:
draw_fig(files[303], cmap='sdoaia171', norm=Normalize(), colorbar=True, process=atrous)

In [ ]:
example = sunpy.map.Map(files[33])

In [ ]:
top_right = SkyCoord(100*u.arcsec, -2900*u.arcsec, frame=example.coordinate_frame)
bottom_left = SkyCoord(-100*u.arcsec, -3100*u.arcsec, frame=example.coordinate_frame)

cut_map = example.submap(bottom_left, top_right=top_right)
draw_fig(cut_map, cmap='sdoaia171', colorbar=True, process=None, norm=Normalize())
plt.savefig('./fig/jet.png', dpi=500)

# 检查出现抖动的图像

In [ ]:
def compare_gif(fsi, hri, filename_gif, process=None, **kwargs):
    """
    将hri图像和fsi图像在一个动图中循环展示，进行肉眼比较
    :param fsi: fsi数据的文件名或sunpy.map.Map对象
    :param hri: hri数据的文件名或sunpy.map.Map对象
    :param filename_gif: 动图的保存位置
    :return: HTML对象，用于在notebook展示动图
    """            
    if type(fsi) == str:
        fsi = sunpy.map.Map(fsi)
    if type(hri) == str:
        hri = sunpy.map.Map(hri)
        
    with plt.ioff():        
        fig_fsi = draw_fig(fsi, process=process, **kwargs)
        fig_fsi.savefig('./fig/fsi.png', dpi=500)
        plt.close(fig_fsi)
        
        fig_hri = draw_fig(hri, process=process, **kwargs)
        fig_hri.savefig('./fig/hri.png', dpi=500)
        plt.close(fig_hri)
    
        image_files = ['./fig/fsi.png', './fig/hri.png']

        images = [Image.open(image) for image in image_files]

        images[0].save(filename_gif, save_all=True, append_images=images[1:], duration=500, loop=0)
        
    current_directory = os.getcwd()
    relative_path = os.path.relpath(filename_gif, current_directory)
    timestamp = int(time.time())
    
    return HTML(f'<img src={relative_path}?{timestamp} height="480">')

In [ ]:
compare_gif(files[31], files[35], './fig/hri_compare.gif', process=atrous, cmap='sdoaia171', norm=Normalize(vmin=-20, vmax=20))

In [ ]:
with plt.ioff(): 
    top_right = (100, -2900)
    bottom_left = (-100, -3100)
    
    cut1 = cut_map(files[31], bottom_left, top_right)
    cut2 = cut_map(files[35], bottom_left, top_right)
    
    fig1 = draw_fig(cut1, process=atrous, cmap='sdoaia171', norm=Normalize(vmin=-12, vmax=12), grid=True)
    fig2 = draw_fig(cut2, process=atrous, cmap='sdoaia171', norm=Normalize(vmin=-12, vmax=12), grid=True)
    
    ax1 = fig1.axes[0]
    ax2 = fig2.axes[0]
    
    r = cut1.fits_header['RSUN_ARC'] * (20 * 1e6 / cut1.fits_header['RSUN_REF'] + 1) / 3600
    
    circle1 = Circle((0, 0), r, color='white', fill=False, linestyle='--', transform=ax1.get_transform('world'))
    circle2 = Circle((0, 0), r, color='white', fill=False, linestyle='--', transform=ax2.get_transform('world'))
    
    ax1.add_patch(circle1)
    ax2.add_patch(circle2)
    
fig1.savefig('./fig/file30.png', dpi=500)
fig2.savefig('./fig/file32.png', dpi=500)

In [ ]:
image_files = ['./fig/file30.png', './fig/file32.png']
images = [Image.open(image) for image in image_files]
images[0].save('./fig/file3032.gif', save_all=True, append_images=images[1:], duration=500, loop=0)
current_directory = os.getcwd()
relative_path = os.path.relpath('./fig/file3032.gif', current_directory)
timestamp = int(time.time())
HTML(f'<img src={relative_path}?{timestamp} height="480">')

## 制作视频查看

In [5]:
deg_range = (179, 182)
d_deg = 0.01
d_time = 3
deg_vect = np.arange(deg_range[0], deg_range[1] + d_deg / 2, d_deg)
time_vect = np.arange(0, len(files) * d_time + 0.5 * d_time, d_time) / 60  # unit: minute
r = 20
fig_st, ax = plt.subplots(figsize=(6, 4))

st_data = Sunlimb.space_time_plot(ax=ax, files=files, deg_range=deg_range, r=r, d_time=3, d_deg=0.01, process=atrous,
                                  cmap='sdoaia171',
                                  title='space_time_plot r=20Mm', xlabel='degree', ylabel='time(min)', norm=Normalize(),
                                  colorbar=True)

<IPython.core.display.Javascript object>

Processing files:   0%|          | 0/200 [00:00<?, ?it/s]

In [10]:
def draw_frame(fig, i, file, deg_vect, time_vect, st_data):
    top_right = (100, -2850)
    bottom_left = (-150, -3100)
    cut = cut_map(file, bottom_left, top_right)
    cut_data = atrous(cut.data)
    cut = sunpy.map.Map(cut_data, cut.fits_header)

    r = cut.fits_header['RSUN_ARC'] * (20 * 1e6 / cut.fits_header['RSUN_REF'] + 1) / 3600
    center = (0, 0)
    begin = -92
    end = -89

    # 创建 ax1 轴并绘制图像
    ax1 = fig.add_subplot(1, 2, 1, projection=cut)
    img_cut = cut.plot(cmap='sdoaia171', norm=Normalize(vmin=-15, vmax=15))  # 图像对象
    wedge = Wedge(center, r, begin, end, color="white", transform=ax1.get_transform('world'), fill=False, linestyle='--')
    ax1.add_patch(wedge)
    ax1.invert_xaxis()
    ax1.invert_yaxis()

    # 创建 ax2 轴并绘制空间时间图像
    ax2 = fig.add_subplot(1, 2, 2)
    Sunlimb.plot_data(ax2, st_data, deg_vect, time_vect, colorbar=True, cmap='sdoaia171', 
                                  title='space_time_plot r=20Mm', xlabel='degree', ylabel='time(min)', norm=Normalize(vmin=-15, vmax=15))
    ax2.hlines(i/20, 179, 182, color='white', linestyle='--')

from tqdm.notebook import tqdm
def update_frame(files, deg_vect, time_vect, st_data):
    fig = plt.figure(figsize=(11, 4))
    for i in tqdm(range(len(files)), total=len(files), desc='drawing...'):
        fig.clear()
        draw_frame(fig, i, files[i], deg_vect, time_vect, st_data)
        _, fits_header = fits.getdata(files[i], header=True)
        filename = fits_header['FILENAME']
        fig.savefig(f'./fig/st_averaged/{filename}.png', dpi=500)

In [11]:
update_frame(files, deg_vect, time_vect, st_data)

<IPython.core.display.Javascript object>

drawing...:   0%|          | 0/200 [00:00<?, ?it/s]

In [14]:
import image
image.images_to_video('./fig/st_averaged/', './fig/st_averaged.mp4', fps=10)

# 尝试其他锐化算法
## sobel算子方法

In [6]:
from scipy import ndimage

example = sunpy.map.Map(files[0])
sx = ndimage.sobel(example.data, axis=0, mode='constant')
sy = ndimage.sobel(example.data, axis=1, mode='constant')
edge_enhanced_im = np.hypot(sx, sy)

edge_map = sunpy.map.Map(edge_enhanced_im, example.meta)

_ = draw_fig(edge_map, cmap='sdoaia171', norm=Normalize(vmax=600), colorbar=True)

<IPython.core.display.Javascript object>

## 拉普拉斯算子方法

In [6]:
def laplace_sharp(image):
    l_image = ndimage.laplace(image)
    return image - l_image

_ = draw_fig(example, cmap='sdoaia171', process=laplace_sharp, norm=Normalize(), colorbar=True)

<IPython.core.display.Javascript object>

In [7]:
def sharp(image):
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]])
    sharp_image = ndimage.convolve(image, kernel)
    return sharp_image

_ = draw_fig(example, cmap='sdoaia171', process=None, norm=Normalize(), colorbar=True)

<IPython.core.display.Javascript object>

In [11]:
top_right = (100, -2850)
bottom_left = (-150, -3100)

example = cut_map(files[0], bottom_left, top_right)
_ = draw_fig(example, cmap='sdoaia171', process=None, norm=Normalize(), colorbar=True)

<IPython.core.display.Javascript object>